# Ministral-8B-Instruct-2410 Inference — NL to CNL Translation


## 0 · Login & Imports

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import torch
import json
import os
from pathlib import Path
from transformers import AutoTokenizer, pipeline
from peft import AutoPeftModelForCausalLM
from tqdm import tqdm
import pandas as pd
import requests
from datasets import load_dataset

import torch
from transformers import LogitsProcessor, PreTrainedTokenizer, AutoTokenizer, AutoModelForCausalLM
from lark import Lark, UnexpectedToken, UnexpectedCharacters, UnexpectedEOF
from abc import ABC, abstractmethod
from typing import Dict


os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Libraries loaded.")
print(f"torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 1 · Load Fine-Tuned Adapter

In [ ]:
# ── Update this path to your saved adapter / best checkpoint ──────────────────
adapter_path = "Path/To/Your/Saved/Adapter"

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,   #Training precision — bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded from : {adapter_path}")
print(f"Model class       : {model.__class__.__name__}")
device_map = getattr(model, 'hf_device_map', None) or getattr(model.base_model, 'hf_device_map', 'N/A')
print(f"Device map        : {device_map}")
print(f"GPU memory        : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 2 · Load Test Dataset

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in Translating the Natural language (NL) into "
    "Controlled Natural Language (CNL) translation. "
    "Always provide precise, syntactically correct translations of NL into CNL."
)

def load_data(path, test_size=0.0, seed=42):
    """
    Load JSON dataset. Dataset and convert to conversational format for NL → CNL.

    Message format:
        user: "{SYSTEM_PROMPT}\n\nTranslate the following...: {NL}"
        assistant: "{CNL}"
    """
    def create_conversation(sample):
        return {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"{SYSTEM_PROMPT}\n\n"
                        f"Translate the following natural language to controlled "
                        f"natural language: {sample['NL_V2']}"
                    )
                },
                {
                    "role": "assistant",
                    "content": sample["CNL_V2"]
                },
            ]
        }

    dataset = load_dataset("json", data_files=path, field="data_dict", split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)

    if test_size == 0:
        return dataset, None

    split = dataset.train_test_split(test_size=test_size, seed=seed)
    return split["train"].shuffle(seed=seed), split["test"]

In [ ]:
dataset_file = "path/to/your/test_dataset.json"
testset, _ = load_data(dataset_file, test_size=0)

print(f"Test dataset size: {len(testset)}")
print("\nExample from test set:")
print(testset[0])

## 3 · Decoder Class

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Abstract base
# ─────────────────────────────────────────────────────────────────────────────
class CNLDecoder(ABC):
    @abstractmethod
    def decode(self, prompt: str, max_new_tokens=100, temperature=0.5) -> str:
        pass


# ─────────────────────────────────────────────────────────────────────────────
# Grammar-constrained logits processor
# Ministral special tokens: <s>, [INST], [/INST], </s>
# ─────────────────────────────────────────────────────────────────────────────
class CandidateGrammarLogitsProcessor(LogitsProcessor):

    def __init__(self, parser, partial_generation, tokenizer, candidate_limit=50):
        self.tokenizer = tokenizer
        self.parser = parser
        self.candidate_limit = candidate_limit
        self.decoded_tokens_cache = self.build_decoded_tokens_cache(tokenizer)
        self.partial_generation = partial_generation
        # Ministral-8B-2410 (V3-Tekken) special tokens
        self.prompt_tokens = [
            "<s>",
            "</s>",
            "[INST]",
            "[/INST]",
        ]

    @staticmethod
    def build_decoded_tokens_cache(tokenizer: PreTrainedTokenizer) -> Dict[int, str]:
        return {token_id: tokenizer.decode(token_id) for _, token_id in tokenizer.get_vocab().items()}

    def is_token_valid(self, text: str) -> bool:
        if not text:
            return False
        try:
            self.parser.parse(text)
            return True
        except UnexpectedEOF:
            return True
        except UnexpectedCharacters:
            return False
        except UnexpectedToken:
            return True

    def __call__(self, input_ids, scores):
        batch_size = scores.shape[0]
        for i in range(batch_size):
            top_k_values, top_k_indices = torch.topk(
                scores[i],
                k=min(self.candidate_limit, scores.shape[-1]),
                dim=-1
            )
            candidates = top_k_indices.tolist()
            valid_candidates = []
            checked_candidates = []

            for token_id in candidates:
                decoded_token = self.decoded_tokens_cache[token_id]
                if decoded_token in self.prompt_tokens or decoded_token.strip() == "be":
                    valid_candidates.append(token_id)
                    continue

                copula_tokens = ["be", "are", "is", "have", "has"]
                decoded_token = decoded_token if decoded_token.strip() not in copula_tokens else decoded_token + " "

                is_valid = self.is_token_valid(self.partial_generation + decoded_token)
                checked_candidates.append(f"-----{self.partial_generation}--: {decoded_token} ----- {is_valid}")
                if is_valid:
                    valid_candidates.append(token_id)

            print("\n".join(checked_candidates))

            if len(valid_candidates) > 0:
                new_scores = torch.full_like(scores[i], -float("inf"))
                for valid_id in valid_candidates:
                    new_scores[valid_id] = scores[i, valid_id]
                scores[i] = new_scores

        return scores


# ─────────────────────────────────────────────────────────────────────────────
# Constrained decoder — no THINK_BOUNDARY_TOKENS, no strip_think_tokens
# ─────────────────────────────────────────────────────────────────────────────
class ConstrainedCNLDecoder(CNLDecoder):

    def __init__(self, model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizer, parser: Lark, candidate_limit=50):
        self.tokenizer = tokenizer
        self.model = model
        self.candidate_limit = candidate_limit
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.parser = parser

    def decode(self, prompt: str, max_new_tokens=512, temperature=0.1) -> str:
        if self.model is None:
            raise ValueError("Model is not loaded.")

        encoded = self.tokenizer(prompt, return_tensors="pt")
        input_ids      = encoded["input_ids"].to(self.device)
        attention_mask = encoded["attention_mask"].to(self.device)

        partial_generation = ""
        num_generated = 0

        stop_ids = self.model.config.eos_token_id
        stop_ids = {stop_ids} if isinstance(stop_ids, int) else set(stop_ids)

        while num_generated < max_new_tokens:
            grammar_processor = CandidateGrammarLogitsProcessor(
                partial_generation=partial_generation,
                tokenizer=self.tokenizer,
                parser=self.parser,
                candidate_limit=self.candidate_limit
            )

            output = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=1,
                do_sample=False,
                temperature=temperature,
                logits_processor=[grammar_processor],
            )

            new_token_id = output[0, -1].unsqueeze(0).unsqueeze(0)
            gen_token    = self.tokenizer.decode(new_token_id[0], skip_special_tokens=False)

            if new_token_id.item() in stop_ids:
                break

            partial_generation += gen_token
            input_ids = torch.cat([input_ids, new_token_id], dim=1)
            attention_mask = torch.cat(
                [attention_mask, torch.ones_like(new_token_id)], dim=1
            )
            num_generated += 1

        return partial_generation.strip()


# ─────────────────────────────────────────────────────────────────────────────
# Naive (unconstrained) decoder
# No strip_think_tokens — Ministral has no thinking mode
# ─────────────────────────────────────────────────────────────────────────────
class NaiveCNLDecoder(CNLDecoder):

    def __init__(self, model, tokenizer: PreTrainedTokenizer):
        self.tokenizer = tokenizer
        self.model     = model
        self.device    = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.pipeline  = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map=self.device
        )

    def decode(self, prompt: str, max_new_tokens=512, temperature=0) -> str:
        outputs = self.pipeline(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=None,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        generated_text = outputs[0]['generated_text']
        prediction     = generated_text[len(prompt):].strip()

        # Remove EOS token if present
        if self.tokenizer.eos_token and self.tokenizer.eos_token in prediction:
            prediction = prediction.split(self.tokenizer.eos_token)[0].strip()

        # Ministral: no think tokens — no strip_think_tokens needed
        return prediction


print("Decoder classes defined.")

## 4 · SyntaxTester (with checkpoint saving)

In [ ]:
SYNTAX_URL = "http://your-server/api/check_syntax" # Replace with your server IP and compile API endpoint
COMPILE_URL = "http://your-server/api/compile"  # Replace with your server IP and compile API endpoint
API_KEY     = "API_KEY_HERE"   # Replace with your actual API key
HEADERS     = {"Content-Type": "application/json", "X-API-KEY": API_KEY}


class SyntaxTester:
    def __init__(self, decoder: CNLDecoder, tokenizer: PreTrainedTokenizer, test_dataset,
                 checkpoint_path="eval_checkpoint_ministral8B.json"):
        self.test_dataset    = test_dataset
        self.decoder         = decoder
        self.tokenizer       = tokenizer
        self.checkpoint_path = checkpoint_path

        if os.path.exists(self.checkpoint_path):
            with open(self.checkpoint_path, 'r') as f:
                checkpoint = json.load(f)
                self.error_count   = checkpoint.get("error_count", 0)
                self.wrong_indexes = checkpoint.get("wrong_indexes", [])
                self.predictions   = checkpoint.get("predictions", [])
                print(f"Resuming from sample {len(self.predictions)}...")
        else:
            self.error_count   = 0
            self.wrong_indexes = []
            self.predictions   = []

    def evaluate(self, verbose=False, save_every=10):
        start_idx = len(self.predictions)

        for idx in tqdm(range(start_idx, len(self.test_dataset)), desc="Evaluating"):
            sample   = self.test_dataset[idx]
            # User content contains: SYSTEM_PROMPT + \n\n + instruction + NL
            # Strip the instruction prefix to get the raw NL input
            user_content = sample["messages"][0]["content"]
            input_nl = user_content.split(
                "Translate the following natural language to controlled natural language: "
            )[-1].strip()

            predicted_cnl = self.__predict(input_nl)

            if verbose:
                print(f"\n[{idx+1}/{len(self.test_dataset)}] NL: {input_nl}")
                print(f"  Predicted CNL: {predicted_cnl}")

            if not self.__syntax_check(predicted_cnl):
                self.error_count += 1
                self.wrong_indexes.append(idx)

            self.predictions.append({
                "input_nl":      input_nl,
                "predicted_cnl": predicted_cnl
            })

            if (idx + 1) % save_every == 0 or (idx + 1) == len(self.test_dataset):
                self._save_progress()

        total_samples = len(self.test_dataset)
        accuracy = (total_samples - self.error_count) / total_samples if total_samples > 0 else 0.0

        if verbose:
            print(f"\nTotal samples : {total_samples}")
            print(f"Syntax errors : {self.error_count}")
            print(f"Accuracy      : {accuracy:.4f}")

        return accuracy

    def _save_progress(self):
        checkpoint = {
            "error_count":   self.error_count,
            "wrong_indexes": self.wrong_indexes,
            "predictions":   self.predictions
        }
        with open(self.checkpoint_path, 'w') as f:
            json.dump(checkpoint, f)

    def __predict(self, input_text: str, max_new_tokens=512) -> str:
        """
        Build inference prompt matching exactly the training format:
        <s>[INST]{SYSTEM_PROMPT}\n\nTranslate the following...: {input}[/INST]

        IMPORTANT: use 'user' role only — no 'system' role.
        The tokenizer silently drops system messages.
        """
        prompt = self.tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": (
                        f"{SYSTEM_PROMPT}\n\n"
                        f"Translate the following natural language to controlled "
                        f"natural language: {input_text}"
                    )
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        return self.decoder.decode(prompt, max_new_tokens=max_new_tokens, temperature=0.1)

    def __syntax_check(self, cnl_text: str) -> bool:
        try:
            response = requests.post(
                SYNTAX_URL,
                headers=HEADERS,
                json={"cnls": cnl_text},
                timeout=10
            )
            response.raise_for_status()
            result = response.json()
            return result.get("cli_message") == "Input file fits the grammar."
        except (requests.exceptions.RequestException, json.JSONDecodeError):
            return False


print("✓ SyntaxTester defined.")

## 5 · Decoder & Run a Single NL Test

In [ ]:
single_NL = "INPUT_NL_HERE"  # Replace with a single natural language sentence for testing

# ── Normal (unconstrained) decoding ──────────────────────────────────────────
decoder: CNLDecoder = NaiveCNLDecoder(model=model, tokenizer=tokenizer)


tester = SyntaxTester(
    decoder=decoder,
    tokenizer=tokenizer,
    test_dataset=[{
        "messages": [
            {
                "role": "user",
                "content": (
                    f"{SYSTEM_PROMPT}\n\n"
                    f"Translate the following natural language to controlled "
                    f"natural language: {single_NL}"
                )
            }
        ]
    }]
)

results = tester.evaluate(verbose=True)
print(f"\nAccuracy: {results}")
wrong_sentences = [tester.predictions[i] for i in tester.wrong_indexes]
print("Predictions:", tester.predictions)
wrong_sentences

## 6 · ASPGenerator 

In [ ]:
class ASPGenerator(SyntaxTester):
    def __init__(self, decoder: CNLDecoder, tokenizer: PreTrainedTokenizer, test_dataset):
        super().__init__(decoder, tokenizer, test_dataset)
        self.data_dict          = []
        self.compilation_errors = 0

    def __get_syntax_status(self, cnl_text: str) -> bool:
        try:
            response = requests.post(
                SYNTAX_URL, headers=HEADERS, json={"cnls": cnl_text}, timeout=10
            )
            result = response.json()
            return result.get("cli_message") == "Input file fits the grammar."
        except Exception:
            return False

    def __get_asp(self, cnl_text: str) -> str:
        try:
            response = requests.post(
                COMPILE_URL, headers=HEADERS, json={"cnls": cnl_text}, timeout=15
            )
            response.raise_for_status()
            result      = response.json()
            asp_content = result.get("asp", "")
            if asp_content and asp_content.strip():
                return asp_content
            self.compilation_errors += 1
            return "ERROR! NOT COMPILING"
        except Exception:
            self.compilation_errors += 1
            return "ERROR! NOT COMPILING"

    def process_and_save(
        self,
        json_path: str,
        output_filename: str = "outputfilename.csv",
        verbose: bool = True
    ):
        with open(json_path, 'r') as f:
            raw_data = json.load(f)

        samples = raw_data.get("data_dict", [])
        self.data_dict          = []
        self.compilation_errors = 0
        total_samples           = len(samples)

        print(f"Starting Ministral-8B Pipeline: processing {total_samples} samples...")

        for item in tqdm(samples, disable=not verbose):
            nl_input   = item.get('NL_V2', '')
            actual_cnl = item.get('CNL_V2', '')
            actual_asp = item.get('ASP', '')
            category   = item.get('Category', 'N/A')
            item_id    = item.get('ID', item.get('Id', 'N/A'))

            # _SyntaxTester__predict uses name-mangled access to parent's __predict
            predicted_cnl   = self._SyntaxTester__predict(nl_input)
            is_valid_syntax = self.__get_syntax_status(predicted_cnl)
            generated_asp   = self.__get_asp(predicted_cnl)

            self.data_dict.append({
                'Natural Language': nl_input,
                'Actual CNL':       actual_cnl,
                'Predicted CNL':    predicted_cnl,
                'Syntax Valid':     is_valid_syntax,
                'Generated ASP':    generated_asp,
                'Actual ASP':       actual_asp,
                'Category':         category,
                'ID':               item_id
            })

        results_df = pd.DataFrame(self.data_dict)
        results_df.to_csv(output_filename, index=False)

        if verbose:
            syntax_acc = (results_df['Syntax Valid'].sum() / total_samples) * 100
            print("\n" + "="*40)
            print("MINISTRAL-8B EVALUATION SUMMARY")
            print("="*40)
            print(f"Total Samples   : {total_samples}")
            print(f"Syntax Accuracy : {syntax_acc:.2f}%")
            print(f"ASP Success Rate: {((total_samples - self.compilation_errors) / total_samples) * 100:.2f}%")
            print(f"CSV saved to    : {output_filename}")
            print("="*40)

        return self.data_dict


print("ASPGenerator defined.")

## 8 · Run Full Evaluation Pipeline

In [ ]:
dataset_file = "path/to/your/test_dataset.json"

# ── Normal (unconstrained) decoding ──────────────────────────────────────────
decoder: CNLDecoder = NaiveCNLDecoder(model=model, tokenizer=tokenizer)

generator = ASPGenerator(decoder=decoder, tokenizer=tokenizer, test_dataset=None)

results = generator.process_and_save(
    json_path=dataset_file,
    output_filename="path/to/your/outputfilename.csv"
)